# Example 4-1: Determining Parallax
### _Fundamentals of Astrodynamics and Applications_, 5th Ed., 2022, pp. 274-275

This notebook demonstrates calculating parallax given the celestial body and date.

## Install and Import Libraries
---

First, install `valladopy` if it doesn't already exist in your environment:

In [1]:
!pip install -r ../valladopy_version.txt

Import the relevant modules:

In [2]:
import numpy as np
import valladopy.constants as const

## Problem Definition
---

GIVEN: &ensp; Neptune on May 14, 1994, r = 29.664361 AU (1 AU = 149,597,870 km)<br>
FIND: &emsp;$\mathscr{P}_{helio}$ and $\mathscr{P}_{geo}$

In [3]:
r_au = 29.664361  # Neptune's distance from the Sun in AU

## Solution
---

We first find the position vectors of Neptune to both the Sun and Earth at the given time. We can retreive this from the [NASA JPL Horizons system](https://ssd.jpl.nasa.gov/horizons) using their public API. This process is outlined in the [JPL Horizons Queries notebook](../JPL_Horizons_Queries.ipynb) (see **State Vectors** section), but for this exercise, we will use the state vectors provided in the textbook example:

In [4]:
rnep_earth = [-1757460712.2509, 3757470099.1416, 1576777174.1537]  # position vector from Neptune to Earth, km
rnep_sun = [-1666604612.0985, 3868340828.5807, 1624846829.1305]    # position vector from Neptune to the Sun, km

Additionally, we define the Neptune to site vector by adding the Neptune to Earth vector with the site vector:

In [5]:
rnep_site = np.array(rnep_earth) + np.array([const.RE, 0, 0])

print(f'Neptune to site vector: {rnep_site} km')

Neptune to site vector: [-1.75745433e+09  3.75747010e+09  1.57677717e+09] km


The heliocentric parallax angle can be found with **Equation 4-29**:

$$
\cos(\mathscr{P}_{helio}) =
\frac{\vec{r}_{Star\oplus} \cdot \vec{r}_{Starc}}
{\left| \vec{r}_{Star\oplus} \right| \left| \vec{r}_{Starc} \right|}
$$

But because the distance to the object is much larger than the central axis, we can use the sine approximation in **Equation 4-30**:

$$
\sin(\mathscr{P}_{helio}) \cong \frac{a}{r_c}
$$

We'll demonstrate both for this exercise:

In [6]:
par_helio_1 = np.arcsin(1 / r_au)
par_helio_2 = np.arccos(np.dot(rnep_sun, rnep_earth) / (np.linalg.norm(rnep_sun) * np.linalg.norm(rnep_earth)))

print(f'parallax angle (approx):\t{np.degrees(par_helio_1):.6f} deg,\t{par_helio_1 / const.ARCSEC2RAD:.4f}"')
print(f'parallax angle (exact):\t\t{np.degrees(par_helio_2):.6f} deg,\t{par_helio_2 / const.ARCSEC2RAD:.4f}"')

parallax angle (approx):	1.931835 deg,	6954.6043"
parallax angle (exact):		1.666447 deg,	5999.2089"


where the central axis is the semimajor axis of the Earth's orbit, 1 AU.

The approximation is close (~16%), but the angle doesn't seem very small. Neptune is relatively "close" to the Earth compared to the distance of most stars from the Earth.

The geocentric parallax is found similarly:

$$
\cos(\mathscr{P}_{geo}) =
\frac{\vec{r}_{\odot} \cdot \vec{r}_c}
{\left| \vec{r}_{\odot} \right| \left| \vec{r}_c \right|}, \quad
\sin(\mathscr{P}_{geo }) \cong \frac{a}{r_c}
$$

In [7]:
par_geo_1 = np.arcsin(const.RE / (r_au * const.AU2KM))
par_geo_2 = np.arccos(np.dot(rnep_earth, rnep_site) / (np.linalg.norm(rnep_earth) * np.linalg.norm(rnep_site)))

print(f'parallax angle (approx):\t{np.degrees(par_geo_1):.6e} deg,\t{par_geo_1 / const.ARCSEC2RAD:.4f}"')
print(f'parallax angle (exact):\t\t{np.degrees(par_geo_2):.6e} deg,\t{par_geo_2 / const.ARCSEC2RAD:.4f}"')

parallax angle (approx):	8.234856e-05 deg,	0.2965"
parallax angle (exact):		7.562526e-05 deg,	0.2723"


This value is much smaller due to the larger distance to the planet compared to the radius of the Earth. It is also within about 9% of the actual value.